In [5]:
from pathlib import Path
import json, statistics, sys, platform, hashlib
from collections import Counter

ROOT = Path.cwd().resolve()
# Dùng notebook trong thư mục notebooks/ vẫn được
if not (ROOT / "scripts" / "preprocess_qa.py").exists() and (ROOT.parent / "scripts" / "preprocess_qa.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "scripts" / "preprocess_qa.py").exists(), "Hãy chạy notebook từ root repository"
sys.path.insert(0, str(ROOT))
RAW = ROOT / "data/raw/viquad"
PROCESSED = ROOT / "data/processed/viquad"
SPLIT = ROOT / "data/splits/viquad_dev_titles.txt"

def read_jsonl(path):
    if not path.is_file():
        raise FileNotFoundError(f"Thiếu {path}; chạy prepare_viquad_qa.py trước")
    with path.open(encoding="utf-8-sig") as f:
        return [json.loads(line) for line in f if line.strip()]

train = read_jsonl(RAW / "train.jsonl")
dev = read_jsonl(RAW / "dev.jsonl")
public_test = read_jsonl(RAW / "public_test.jsonl")
assert SPLIT.exists(), f"Thiếu danh sách title Dev: {SPLIT}"
expected_titles = {line.strip() for line in SPLIT.read_text(encoding="utf-8-sig").splitlines() if line.strip()}
actual_titles = {row["title"] for row in dev}
train_titles = {row["title"] for row in train}
assert len(expected_titles) == 13, f"Không đúng 13 title: {len(expected_titles)}"
assert actual_titles == expected_titles, f"Dev titles khác file split; thiếu {expected_titles - actual_titles}, thừa {actual_titles - expected_titles}"
assert not (train_titles & actual_titles), f"Title leakage: {train_titles & actual_titles}"
print('PASS: split 13 title, train/dev không trùng title')
print('Python:', platform.python_version())
print('Repo:', ROOT)

PASS: split 13 title, train/dev không trùng title
Python: 3.12.8
Repo: C:\Users\DELL\Downloads\DL-Group09-vietnamese-qa-rag\DL-Group09-vietnamese-qa-rag


In [6]:
def profile(name, rows):
    lengths = [len(row.get("context") or "") for row in rows]
    questions = [len(row.get("question") or "") for row in rows]
    counts = Counter(row.get("is_impossible") for row in rows)
    return {
        "split": name,
        "questions": len(rows),
        "titles": len({row.get("title") for row in rows}),
        "unique_contexts": len({(row.get("title"), row.get("context")) for row in rows}),
        "answerable": counts.get(False, 0),
        "impossible": counts.get(True, 0),
        "impossible_pct": round(100 * counts.get(True, 0) / max(1, len(rows)), 2),
        "context_chars_median": int(statistics.median(lengths)),
        "context_chars_max": max(lengths),
        "question_chars_median": int(statistics.median(questions)),
    }

profiles = [profile(n, r) for n,r in [('train',train),('dev',dev),('public_test',public_test)]]
for item in profiles:
    print(json.dumps(item, ensure_ascii=False))

# Validate answer offset for train/dev (raw, trước sửa lỗi)
def offset_audit(rows):
    correct = mismatch = missing = 0
    example_errors=[]
    for row in rows:
        if row.get("is_impossible"):
            continue
        answers = row.get("answers") or {}
        spans = list(zip(answers.get("text",[]), answers.get("answer_start",[])))
        if not spans:
            missing += 1
        for ans, start in spans:
            if not isinstance(start,int) or row["context"][start:start+len(ans)] != ans:
                mismatch += 1
                if len(example_errors)<5: example_errors.append(row.get("id"))
            else:
                correct += 1
    return {"matching_spans":correct,"mismatched_spans":mismatch,"missing_answer_records":missing,"example_error_ids":example_errors}

audit = {"train":offset_audit(train),"dev":offset_audit(dev)}
print('Answer offset audit (RAW):',json.dumps(audit,ensure_ascii=False,indent=2))
if PROCESSED.exists() and all((PROCESSED/f'{s}.jsonl').is_file() for s in ['train','dev']):
    processed_audit={s:offset_audit(read_jsonl(PROCESSED/f'{s}.jsonl')) for s in ['train','dev']}
    print('Answer offset audit (PROCESSED):', json.dumps(processed_audit,ensure_ascii=False))
else:
    processed_audit=None
    print('Chưa có processed train/dev để audit answer_start sau hiệu chỉnh.')

{"split": "train", "questions": 25686, "titles": 125, "unique_contexts": 3711, "answerable": 17352, "impossible": 8334, "impossible_pct": 32.45, "context_chars_median": 741, "context_chars_max": 7221, "question_chars_median": 63}
{"split": "dev", "questions": 2768, "titles": 13, "unique_contexts": 390, "answerable": 1886, "impossible": 882, "impossible_pct": 31.86, "context_chars_median": 760, "context_chars_max": 2287, "question_chars_median": 60}
{"split": "public_test", "questions": 3814, "titles": 19, "unique_contexts": 557, "answerable": 2653, "impossible": 1161, "impossible_pct": 30.44, "context_chars_median": 721, "context_chars_max": 2852, "question_chars_median": 62}
Answer offset audit (RAW): {
  "train": {
    "matching_spans": 17206,
    "mismatched_spans": 146,
    "missing_answer_records": 0,
    "example_error_ids": [
      "0004-0002-0003",
      "0004-0005-0003",
      "0006-0002-0001",
      "0011-0005-0001",
      "0011-0007-0005"
    ]
  },
  "dev": {
    "matching_

In [ ]:
import random
from src.preprocessing_qa import align_tokens_to_text, preprocess_example, MAX_LEN, DOC_STRIDE
vncorenlp_result = {"status":"not_run", "success":0,"failed":0,"failures":[]}
try:
    import py_vncorenlp
    from transformers import AutoTokenizer
    model_dir = ROOT / 'data/raw/vncorenlp'
    if not model_dir.is_dir():
        raise FileNotFoundError(f'Chưa có VnCoreNLP models: {model_dir}')
    segmenter = py_vncorenlp.VnCoreNLP(annotators=['wseg'],save_dir=str(model_dir))
    tokenizer = AutoTokenizer.from_pretrained('vinai/phobert-base', use_fast=False)

    def segment(text):
        tokens = ' '.join(segmenter.word_segment(text)).split()
        if '𨮁' in text:
            tokens = ['𨮁' if token == '讁' else token for token in tokens]
        return tokens

    rng = random.Random(42)
    processed_dev = read_jsonl(PROCESSED / "dev.jsonl")
    answerable = [r for r in processed_dev if not r["is_impossible"]]
    impossible = [r for r in processed_dev if r["is_impossible"]]
    samples = rng.sample(answerable, min(5,len(answerable))) + rng.sample(impossible, min(5,len(impossible)))
    vncorenlp_result.update({"status":"executed","sample_size":len(samples),"max_len":MAX_LEN,"doc_stride":DOC_STRIDE})
    for row in samples:
        try:
            question_tokens=segment(row['question'])
            context_tokens=segment(row['context'])
            aligned=align_tokens_to_text(row['context'],context_tokens)
            features = preprocess_example(
                row,
                question_tokens,
                context_tokens,
                tokenizer,
                mode="eval",
            )
            assert features and all(len(x['input_ids'])<=MAX_LEN for x in features)
            pos=sum(x['start_positions']!=0 for x in features)
            if not row['is_impossible']:
                assert pos>=1, 'Answerable không có positive window'
            vncorenlp_result['success']+=1
            if vncorenlp_result['success']<=3:
                print('ID:',row['id'],'| word tokens:',len(context_tokens),'| windows:',len(features),'| positive:',pos)
                print('VnCoreNLP:', context_tokens[:12])
                print('PhoBERT subwords:', tokenizer.tokenize(' '.join(context_tokens[:12]))[:24])
                print('Token spans ký tự:',aligned[:5])
        except Exception as e:
            vncorenlp_result['failed']+=1
            vncorenlp_result['failures'].append({'id':row.get('id'),'error':f'{type(e).__name__}: {e}'})
    print('VnCoreNLP/PhoBERT:',json.dumps(vncorenlp_result,ensure_ascii=False,indent=2))
except (ImportError,FileNotFoundError,RuntimeError,OSError) as e:
    vncorenlp_result['status']='unavailable'
    vncorenlp_result['reason']=f'{type(e).__name__}: {e}'
    print('CHƯA XÁC MINH tokenizer tại môi trường này:',vncorenlp_result['reason'])

CHƯA XÁC MINH tokenizer tại môi trường này: ModuleNotFoundError: No module named 'py_vncorenlp'


In [8]:
import datetime
out = ROOT / 'docs/qa_week2_A_eda_reproduced.md'
out.parent.mkdir(parents=True, exist_ok=True)
rows = ['# Tuần 2 – A: EDA ViQuAD và VnCoreNLP/PhoBERT',
        f'- Ngày tạo: {datetime.datetime.now().isoformat(timespec="seconds")}',
        '- Dữ liệu: data/raw/viquad/{train,dev,public_test}.jsonl',
        '- Split: data/splits/viquad_dev_titles.txt (13 title, PASS; Train/Dev title overlap = 0)',
        '- Khuyến nghị: chỉ dùng Dev để tuning, không đánh giá public_test trước mốc test cuối.',
        '', '## EDA',
        '| Split | Questions | Titles | Contexts | Answerable | Impossible | Impossible % | Median context chars |',
        '|---|---:|---:|---:|---:|---:|---:|---:|']
for p in profiles:
    rows.append(f"| {p['split']} | {p['questions']} | {p['titles']} | {p['unique_contexts']} | {p['answerable']} | {p['impossible']} | {p['impossible_pct']} | {p['context_chars_median']} |")
rows += ['', '## Answer offsets (RAW)', '```json',json.dumps(audit,ensure_ascii=False,indent=2),'```',
         '', '## Answer offsets (PROCESSED)', '```json',json.dumps(processed_audit,ensure_ascii=False,indent=2),'```',
         '', '## VnCoreNLP và PhoBERT tokenizer', '```json',json.dumps(vncorenlp_result,ensure_ascii=False,indent=2),'```',
         '', '## Kết luận', '- Kiểm tra 13 title và không overlap: PASS',
         '- VnCoreNLP/PhoBERT: '+ ('đã kiểm tra mẫu' if vncorenlp_result['status']=='executed' else 'CHƯA XÁC MINH trên môi trường này'),
         '- Kiểm tra log tiền xử lý toàn bộ: xem data/processed/qa_features/pipeline_summary.json (nếu đã chạy).']
out.write_text('\n'.join(rows)+'\n',encoding='utf-8')
print('Đã tạo:',out)

Đã tạo: C:\Users\DELL\Downloads\DL-Group09-vietnamese-qa-rag\DL-Group09-vietnamese-qa-rag\docs\qa_week2_A_eda_reproduced.md
